In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the dataset
df = pd.read_csv('../data/mental_health_social_media_dataset.csv')
print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

Dataset shape: (5000, 15)
Columns: ['person_name', 'age', 'date', 'gender', 'platform', 'daily_screen_time_min', 'social_media_time_min', 'negative_interactions_count', 'positive_interactions_count', 'sleep_hours', 'physical_activity_min', 'anxiety_level', 'stress_level', 'mood_level', 'mental_state']


,person_name,age,date,gender,platform,daily_screen_time_min,social_media_time_min,negative_interactions_count,positive_interactions_count,sleep_hours,physical_activity_min,anxiety_level,stress_level,mood_level,mental_state
0,Reyansh Ghosh,35,1/1/2024,Male,Instagram,320,160,1,2,7.4,28,2,7,6,Stressed
1,Neha Patel,24,1/12/2024,Female,Instagram,453,226,1,3,6.7,15,3,8,5,Stressed
2,Ananya Naidu,26,1/6/2024,Male,Snapchat,357,196,1,2,7.2,24,3,7,6,Stressed
3,Neha Das,66,1/17/2024,Female,Snapchat,190,105,0,1,8.0,41,2,6,6,Stressed
4,Reyansh Banerjee,31,1/28/2024,Male,Snapchat,383,211,1,2,7.1,22,3,7,6,Stressed


In [2]:
# Check for missing values
df.isnull().sum()

person_name                    0
age                            0
date                           0
gender                         0
platform                       0
daily_screen_time_min          0
social_media_time_min          0
negative_interactions_count    0
positive_interactions_count    0
sleep_hours                    0
physical_activity_min          0
anxiety_level                  0
stress_level                   0
mood_level                     0
mental_state                   0
dtype: int64

In [3]:
# Data types
df.dtypes

person_name                     object
age                              int64
date                            object
gender                          object
platform                        object
daily_screen_time_min            int64
social_media_time_min            int64
negative_interactions_count      int64
positive_interactions_count      int64
sleep_hours                    float64
physical_activity_min            int64
anxiety_level                    int64
stress_level                     int64
mood_level                       int64
mental_state                    object
dtype: object

In [4]:
# Basic statistics
df.describe()

,age,daily_screen_time_min,social_media_time_min,negative_interactions_count,positive_interactions_count,sleep_hours,physical_activity_min,anxiety_level,stress_level,mood_level
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,29.947800,373.058200,175.331600,0.864200,1.835400,7.134660,22.693400,2.510400,7.107600,5.625800
std,12.279936,106.003916,71.209329,0.555176,0.943443,0.533184,10.602862,0.794996,1.062378,0.759928
min,13.000000,140.000000,35.000000,0.000000,0.000000,6.400000,8.000000,1.000000,5.000000,4.000000
25%,21.000000,310.000000,118.000000,1.000000,1.000000,6.700000,14.000000,2.000000,6.000000,5.000000
50%,27.000000,388.000000,170.000000,1.000000,2.000000,7.100000,21.000000,3.000000,7.000000,6.000000
75%,35.250000,461.000000,231.000000,1.000000,2.000000,7.450000,29.000000,3.000000,8.000000,6.000000
max,69.000000,520.000000,338.000000,2.000000,4.000000,8.300000,46.000000,4.000000,9.000000,7.000000


In [5]:
# Clean data - drop rows with missing mental_state
df = df.dropna(subset=['mental_state'])
print('After cleaning shape:', df.shape)

After cleaning shape: (5000, 15)


In [15]:
# EDA: Mental State Distribution
fig = px.pie(df, names='mental_state', title='Mental State Distribution', template='plotly_dark')
fig.write_image('../viz/mental_state_distribution.png')
fig.show()

In [16]:
# EDA: Age vs Anxiety Level
fig = px.scatter(df, x='age', y='anxiety_level', color='mental_state', title='Age vs Anxiety Level by Mental State', template='plotly_dark')
fig.write_image('../viz/age_vs_anxiety.png')
fig.show()

In [17]:
# EDA: Screen Time vs Mental State
fig = px.box(df, x='mental_state', y='daily_screen_time_min', title='Daily Screen Time by Mental State', template='plotly_dark')
fig.write_image('../viz/screen_time_vs_mental_state.png')
fig.show()

In [18]:
# EDA: Platform Usage by Mental State
platform_mental = df.groupby(['platform', 'mental_state']).size().reset_index(name='count')
fig = px.bar(platform_mental, x='platform', y='count', color='mental_state', title='Platform Usage by Mental State', template='plotly_dark')
fig.write_image('../viz/platform_vs_mental_state.png')
fig.show()

In [19]:
# EDA: Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()
fig = px.imshow(corr_matrix, text_auto=True, title='Correlation Heatmap', template='plotly_dark')
fig.write_image('../viz/correlation_heatmap.png')
fig.show()

In [11]:
# Preprocessing
# Encode categorical variables
le_gender = LabelEncoder()
df['gender_encoded'] = le_gender.fit_transform(df['gender'])

le_platform = LabelEncoder()
df['platform_encoded'] = le_platform.fit_transform(df['platform'])

# Select features
features = ['age', 'gender_encoded', 'platform_encoded', 'daily_screen_time_min', 'social_media_time_min', 
           'negative_interactions_count', 'positive_interactions_count', 'sleep_hours', 'physical_activity_min']
X = df[features]
y = df['mental_state']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
# Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = rf_model.predict(X_test_scaled)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig = px.imshow(cm, text_auto=True, title='Confusion Matrix', template='plotly_dark',
                labels=dict(x="Predicted", y="Actual"))
fig.write_image('../viz/confusion_matrix.png')

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

     At_Risk       1.00      1.00      1.00        12
     Healthy       1.00      1.00      1.00        73
    Stressed       1.00      1.00      1.00       915

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [13]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

fig = px.bar(feature_importance, x='importance', y='feature', orientation='h', title='Feature Importance', template='plotly_dark')
fig.write_image('../viz/feature_importance.png')

In [ ]:
# Create a pipeline for the model
from sklearn.pipeline import Pipeline

# Define the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Example prediction
# Create a sample mental health data
sample_data = pd.DataFrame({
    'age': [25],
    'gender_encoded': [le_gender.transform(['Female'])[0]],
    'platform_encoded': [le_platform.transform(['Instagram'])[0]],
    'daily_screen_time_min': [400],
    'social_media_time_min': [200],
    'negative_interactions_count': [5],
    'positive_interactions_count': [10],
    'sleep_hours': [7.5],
    'physical_activity_min': [30]
})

# Make prediction
prediction = pipeline.predict(sample_data)
prediction_proba = pipeline.predict_proba(sample_data)

print('Sample Mental Health Prediction:')
print(f'Predicted Mental State: {prediction[0]}')
print(f'Prediction Probabilities: {prediction_proba[0]}')

# Save the pipeline
import joblib
joblib.dump(pipeline, '../models/mental_health_pipeline.joblib')
print('Pipeline saved to ../models/mental_health_pipeline.joblib')

Sample Mental Health Prediction:
Predicted Mental State: Stressed
Prediction Probabilities: [0. 0. 1.]
Pipeline saved to ../models/mental_health_pipeline.joblib
